# Notebook 3 - DINOv2 Classification

This notebook runs the DINOv2 linear-probe and fine-tuning experiments.
It reuses the shared split metadata and saved embeddings from the project
workspace.

In [ ]:
#@title 1. Setup
from pathlib import Path
import sys

repo_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/content/wild_animal_image_classification'),
    Path('/content'),
]
repo_root = None
for candidate in repo_candidates:
    if (candidate / 'configs' / 'config.py').exists():
        repo_root = candidate
        break
if repo_root is None:
    raise FileNotFoundError('Could not locate the repository root.')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import seaborn as sns
import torch
import torch.nn as nn
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

from configs.config import CFG
from utils.dataset import IWildCamDataset, get_transforms

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
#@title 2. Load Pre-Saved Splits
CFG.ensure_output_dirs()
SPLITS_DIR = CFG.splits_dir
RESULTS_DIR = CFG.results_dir
CKPT_DIR = CFG.checkpoints_dir
EMBED_DIR = CFG.embeddings_dir

train_df = pd.read_csv(f'{SPLITS_DIR}/train.csv')
val_df = pd.read_csv(f'{SPLITS_DIR}/val.csv')
test_df = pd.read_csv(f'{SPLITS_DIR}/test.csv')

with open(f'{SPLITS_DIR}/metadata.json') as f:
    meta = json.load(f)
NUM_CLASSES = meta['num_classes']

with open(f'{SPLITS_DIR}/cat_names.json') as f:
    cat_names = json.load(f)
with open(f'{SPLITS_DIR}/label_to_cat.json') as f:
    label_to_cat = json.load(f)

class_names = [cat_names.get(str(label_to_cat[str(i)]), f'class_{i}') for i in range(NUM_CLASSES)]
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)} | Classes: {NUM_CLASSES}')

In [ ]:
#@title 3. Dataset Settings
IMG_SIZE = CFG.img_size_dino
BATCH_SIZE_LP = CFG.dino_lp_batch_size
BATCH_SIZE_FT = CFG.dino_ft_batch_size
NUM_WORKERS = CFG.num_workers

train_transform = get_transforms(IMG_SIZE, is_train=True)
eval_transform = get_transforms(IMG_SIZE, is_train=False)

In [ ]:
#@title 4. Load DINOv2 ViT-L Backbone
# DINOv2 is loaded via torch.hub (facebookresearch/dinov2)
dino_backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14')
dino_backbone = dino_backbone.to(DEVICE)
dino_backbone.eval()

# DINOv2 ViT-L output dimension
EMBED_DIM = 1024  # dinov2_vitl14 outputs 1024-d CLS token

total_params = sum(p.numel() for p in dino_backbone.parameters())
print(f'DINOv2 ViT-L/14: {total_params/1e6:.1f}M params')
print(f'Embedding dim: {EMBED_DIM}')

---
## Part A: Extract & Save Embeddings
Run DINOv2 inference on all images ONCE, save embeddings to disk.
Both linear probe and Carlos's clustering notebook reuse these.

In [ ]:
#@title 5. Extract Embeddings (run once, ~1-2 hrs on T4)
def extract_embeddings(backbone, df, transform, batch_size=64, device='cuda'):
    """Extract DINOv2 CLS token embeddings for all images."""
    ds = IWildCamDataset(df, transform)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

    all_embeddings = []
    all_labels = []

    backbone.eval()
    with torch.no_grad():
        for i, (images, labels) in enumerate(loader):
            images = images.to(device, non_blocking=True)
            with autocast():
                features = backbone(images)  # CLS token: (B, 1024)
            all_embeddings.append(features.cpu().float().numpy())
            all_labels.extend(labels.numpy())

            if (i + 1) % 50 == 0:
                print(f'  Batch {i+1}/{len(loader)}')

    embeddings = np.concatenate(all_embeddings, axis=0)
    labels = np.array(all_labels)
    return embeddings, labels

# Check if already extracted
train_emb_path = f'{EMBED_DIR}/dinov2_train_embeddings.npz'

if os.path.exists(train_emb_path):
    print('Embeddings already exist — loading from disk.')
    d = np.load(train_emb_path)
    train_emb, train_lbl = d['embeddings'], d['labels']
    d = np.load(f'{EMBED_DIR}/dinov2_val_embeddings.npz')
    val_emb, val_lbl = d['embeddings'], d['labels']
    d = np.load(f'{EMBED_DIR}/dinov2_test_embeddings.npz')
    test_emb, test_lbl = d['embeddings'], d['labels']
else:
    print('Extracting train embeddings...')
    train_emb, train_lbl = extract_embeddings(dino_backbone, train_df, eval_transform)
    np.savez(train_emb_path, embeddings=train_emb, labels=train_lbl)

    print('Extracting val embeddings...')
    val_emb, val_lbl = extract_embeddings(dino_backbone, val_df, eval_transform)
    np.savez(f'{EMBED_DIR}/dinov2_val_embeddings.npz', embeddings=val_emb, labels=val_lbl)

    print('Extracting test embeddings...')
    test_emb, test_lbl = extract_embeddings(dino_backbone, test_df, eval_transform)
    np.savez(f'{EMBED_DIR}/dinov2_test_embeddings.npz', embeddings=test_emb, labels=test_lbl)

    print(f'Saved embeddings to {EMBED_DIR}/')

print(f'Train: {train_emb.shape} | Val: {val_emb.shape} | Test: {test_emb.shape}')

---
## Part B: Linear Probe (frozen backbone)
Train a simple linear layer on top of frozen DINOv2 features.

In [ ]:
#@title 6. Linear Probe Training
from torch.utils.data import TensorDataset

# Convert to tensors
train_tensor_ds = TensorDataset(
    torch.FloatTensor(train_emb), torch.LongTensor(train_lbl)
)
val_tensor_ds = TensorDataset(
    torch.FloatTensor(val_emb), torch.LongTensor(val_lbl)
)
test_tensor_ds = TensorDataset(
    torch.FloatTensor(test_emb), torch.LongTensor(test_lbl)
)

# Weighted sampler for linear probe
counts_lp = Counter(train_lbl.tolist())
weights_lp = [1.0 / counts_lp[int(l)] for l in train_lbl]
sampler_lp = WeightedRandomSampler(weights_lp, num_samples=len(weights_lp), replacement=True)

lp_train_loader = DataLoader(train_tensor_ds, batch_size=256, sampler=sampler_lp)
lp_val_loader = DataLoader(val_tensor_ds, batch_size=256, shuffle=False)
lp_test_loader = DataLoader(test_tensor_ds, batch_size=256, shuffle=False)

# Simple linear head
linear_head = nn.Sequential(
    nn.LayerNorm(EMBED_DIM),
    nn.Linear(EMBED_DIM, NUM_CLASSES)
).to(DEVICE)

lp_optimizer = torch.optim.AdamW(linear_head.parameters(), lr=1e-3, weight_decay=1e-4)
lp_criterion = nn.CrossEntropyLoss()
LP_EPOCHS = 20

print('Training linear probe on frozen DINOv2 embeddings...')
lp_history = {'val_f1': []}
best_lp_f1 = 0.0

for epoch in range(1, LP_EPOCHS + 1):
    linear_head.train()
    for emb, lbl in lp_train_loader:
        emb, lbl = emb.to(DEVICE), lbl.to(DEVICE)
        logits = linear_head(emb)
        loss = lp_criterion(logits, lbl)
        lp_optimizer.zero_grad()
        loss.backward()
        lp_optimizer.step()

    # Validate
    linear_head.eval()
    preds, lbls = [], []
    with torch.no_grad():
        for emb, lbl in lp_val_loader:
            emb = emb.to(DEVICE)
            logits = linear_head(emb)
            preds.extend(logits.argmax(1).cpu().numpy())
            lbls.extend(lbl.numpy())

    val_f1 = f1_score(lbls, preds, average='macro', zero_division=0)
    val_acc = accuracy_score(lbls, preds)
    lp_history['val_f1'].append(val_f1)

    print(f'  Epoch {epoch:>2}/{LP_EPOCHS} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}')

    if val_f1 > best_lp_f1:
        best_lp_f1 = val_f1
        torch.save(linear_head.state_dict(), f'{CKPT_DIR}/dinov2_linear_probe_best.pth')

print(f'\nBest Linear Probe Val F1: {best_lp_f1:.4f}')

In [ ]:
#@title 7. Linear Probe — Test Evaluation
linear_head.load_state_dict(torch.load(f'{CKPT_DIR}/dinov2_linear_probe_best.pth'))
linear_head.eval()

test_preds, test_lbls = [], []
with torch.no_grad():
    for emb, lbl in lp_test_loader:
        emb = emb.to(DEVICE)
        logits = linear_head(emb)
        test_preds.extend(logits.argmax(1).cpu().numpy())
        test_lbls.extend(lbl.numpy())

lp_acc = accuracy_score(test_lbls, test_preds)
lp_f1 = f1_score(test_lbls, test_preds, average='macro', zero_division=0)
lp_f1_w = f1_score(test_lbls, test_preds, average='weighted', zero_division=0)

print(f'\n{"="*50}')
print(f'  DINOv2 Linear Probe — TEST RESULTS')
print(f'{"="*50}')
print(f'  Accuracy:      {lp_acc:.4f}')
print(f'  F1 (macro):    {lp_f1:.4f}')
print(f'  F1 (weighted): {lp_f1_w:.4f}')
print(f'{"="*50}\n')

print(classification_report(test_lbls, test_preds, target_names=class_names, zero_division=0))

---
## Part C: Fine-Tune Last 4 Blocks
Unfreeze the last N transformer blocks of DINOv2 for better performance.

**Requires more VRAM** — if on free Colab T4, reduce `BATCH_SIZE_FT` to 4.

In [ ]:
#@title 8. Build DINOv2 + Classification Head (Fine-tune)
class DINOv2Classifier(nn.Module):
    def __init__(self, backbone, embed_dim, num_classes, unfreeze_blocks=4):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.3),
            nn.Linear(embed_dim, num_classes)
        )

        # Freeze everything first
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Unfreeze last N blocks
        total_blocks = len(self.backbone.blocks)
        for i in range(total_blocks - unfreeze_blocks, total_blocks):
            for param in self.backbone.blocks[i].parameters():
                param.requires_grad = True

        # Unfreeze norm
        for param in self.backbone.norm.parameters():
            param.requires_grad = True

    def forward(self, x):
        features = self.backbone(x)  # CLS token
        return self.head(features)

UNFREEZE_BLOCKS = 4
ft_model = DINOv2Classifier(
    dino_backbone, EMBED_DIM, NUM_CLASSES, unfreeze_blocks=UNFREEZE_BLOCKS
).to(DEVICE)

trainable = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in ft_model.parameters())
print(f'DINOv2 Fine-tune: {total/1e6:.1f}M total, {trainable/1e6:.1f}M trainable '
      f'(last {UNFREEZE_BLOCKS} blocks unfrozen)')

In [ ]:
#@title 9. Fine-tune DataLoaders & Optimizer
BATCH_SIZE_FT = 8  # reduce to 4 if OOM on T4
FT_LR = 1e-5
FT_EPOCHS = 10
FT_GRAD_ACCUM = 4  # effective batch = 8 * 4 = 32
PATIENCE = 5

# Weighted sampler
counts_ft = Counter(train_df['label'].tolist())
weights_ft = [1.0 / counts_ft[label] for label in train_df['label']]
sampler_ft = WeightedRandomSampler(weights_ft, num_samples=len(weights_ft), replacement=True)

ft_train_ds = IWildCamDataset(train_df, train_transform)
ft_val_ds = IWildCamDataset(val_df, eval_transform)
ft_test_ds = IWildCamDataset(test_df, eval_transform)

ft_train_loader = DataLoader(ft_train_ds, batch_size=BATCH_SIZE_FT, sampler=sampler_ft,
                             num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
ft_val_loader = DataLoader(ft_val_ds, batch_size=BATCH_SIZE_FT, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
ft_test_loader = DataLoader(ft_test_ds, batch_size=BATCH_SIZE_FT, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)

ft_criterion = nn.CrossEntropyLoss()
ft_optimizer = torch.optim.AdamW([
    {'params': [p for p in ft_model.backbone.parameters() if p.requires_grad], 'lr': FT_LR},
    {'params': ft_model.head.parameters(), 'lr': FT_LR * 10},
], weight_decay=0.01)
ft_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(ft_optimizer, T_max=FT_EPOCHS)
ft_scaler = GradScaler()

In [ ]:
#@title 10. Fine-tune Training Loop
ft_history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
best_ft_f1 = 0.0
patience_counter = 0
CKPT_FT_PATH = f'{CKPT_DIR}/dinov2_finetune_best.pth'

for epoch in range(1, FT_EPOCHS + 1):
    t0 = time.time()
    ft_model.train()
    running_loss = 0.0
    ft_optimizer.zero_grad()

    for i, (images, labels) in enumerate(ft_train_loader):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with autocast():
            outputs = ft_model(images)
            loss = ft_criterion(outputs, labels) / FT_GRAD_ACCUM

        ft_scaler.scale(loss).backward()
        if (i + 1) % FT_GRAD_ACCUM == 0:
            ft_scaler.unscale_(ft_optimizer)
            torch.nn.utils.clip_grad_norm_(ft_model.parameters(), max_norm=1.0)
            ft_scaler.step(ft_optimizer)
            ft_scaler.update()
            ft_optimizer.zero_grad()

        running_loss += loss.item() * FT_GRAD_ACCUM

    train_loss = running_loss / len(ft_train_loader)

    # Validate
    ft_model.eval()
    val_preds, val_labels_all = [], []
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in ft_val_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with autocast():
                outputs = ft_model(images)
                loss = ft_criterion(outputs, labels)
            val_loss += loss.item()
            val_preds.extend(outputs.argmax(1).cpu().numpy())
            val_labels_all.extend(labels.cpu().numpy())

    val_loss /= len(ft_val_loader)
    val_acc = accuracy_score(val_labels_all, val_preds)
    val_f1 = f1_score(val_labels_all, val_preds, average='macro', zero_division=0)

    ft_scheduler.step()
    elapsed = time.time() - t0

    ft_history['train_loss'].append(train_loss)
    ft_history['val_loss'].append(val_loss)
    ft_history['val_acc'].append(val_acc)
    ft_history['val_f1'].append(val_f1)

    print(f'Epoch {epoch:>2}/{FT_EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | '
          f'Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f} | {elapsed:.1f}s')

    if val_f1 > best_ft_f1:
        best_ft_f1 = val_f1
        patience_counter = 0
        torch.save(ft_model.state_dict(), CKPT_FT_PATH)
        print(f'  -> New best F1: {best_ft_f1:.4f}')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

In [ ]:
#@title 11. Fine-tune — Test Evaluation
ft_model.load_state_dict(torch.load(CKPT_FT_PATH))
ft_model.eval()

test_preds, test_labels_all = [], []
with torch.no_grad():
    for images, labels in ft_test_loader:
        images = images.to(DEVICE, non_blocking=True)
        with autocast():
            outputs = ft_model(images)
        test_preds.extend(outputs.argmax(1).cpu().numpy())
        test_labels_all.extend(labels.numpy())

ft_acc = accuracy_score(test_labels_all, test_preds)
ft_f1 = f1_score(test_labels_all, test_preds, average='macro', zero_division=0)
ft_f1_w = f1_score(test_labels_all, test_preds, average='weighted', zero_division=0)

print(f'\n{"="*50}')
print(f'  DINOv2 Fine-tune (last {UNFREEZE_BLOCKS} blocks) — TEST')
print(f'{"="*50}')
print(f'  Accuracy:      {ft_acc:.4f}')
print(f'  F1 (macro):    {ft_f1:.4f}')
print(f'  F1 (weighted): {ft_f1_w:.4f}')
print(f'{"="*50}\n')

print(classification_report(test_labels_all, test_preds, target_names=class_names, zero_division=0))

# Confusion matrix
cm = confusion_matrix(test_labels_all, test_preds)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'DINOv2 Fine-tune (last {UNFREEZE_BLOCKS} blocks) — Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
fig.savefig(f'{RESULTS_DIR}/dinov2_finetune_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
#@title 12. Save All Results
# Linear probe results
lp_results = {
    'model': 'DINOv2 Linear Probe',
    'accuracy': float(lp_acc),
    'f1_macro': float(lp_f1),
    'f1_weighted': float(lp_f1_w),
}
with open(f'{RESULTS_DIR}/dinov2_linear_probe_final.json', 'w') as f:
    json.dump(lp_results, f, indent=2)

# Fine-tune results
ft_results = {
    'model': f'DINOv2 Fine-tune (last {UNFREEZE_BLOCKS} blocks)',
    'accuracy': float(ft_acc),
    'f1_macro': float(ft_f1),
    'f1_weighted': float(ft_f1_w),
    'best_val_f1': float(best_ft_f1),
    'epochs_trained': len(ft_history['train_loss']),
}
with open(f'{RESULTS_DIR}/dinov2_finetune_final.json', 'w') as f:
    json.dump(ft_results, f, indent=2)

print('All DINOv2 results saved!')